In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.systemModel                  import SystemModel as systemModel
# from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-12-16 22:17:35.572702: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-16 22:17:35.909961: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-16 22:17:35.909981: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-12-16 22:17:35.964859: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-16 22:17:37.239919: W tensorflow/stream_executor/platform/de

##### HI COMPONENT AND SYSTEM

In [2]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

cr1, cr2, cv2           = preData.loadH5File(name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file



# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(cr1)
withHourCr2             = preData.addHours(cr2)
withHourCv2             = preData.addHours(cv2)


# --------------------------------
# Use for system HI
# --------------------------------
insertedCr1Sys, insertedCr2Sys,  insertedCv2Sys     = preData.insertForTheSameLength(data         = [withHourCr1.copy(),withHourCr2.copy(),withHourCv2.copy()], 
                                                                                    validColumn  = [preData.crusher1Value, preData.crusher2Value, preData.conveyorValue + preData.crusher2ConveyorValue])

withIDCr1, withIDCr2, withIDCv2, IdSys  = preData.attachIDPerEvent(df = [insertedCr1Sys,insertedCr2Sys,insertedCv2Sys], mainternanceAsCriticalSd = True)

# insertedCr1 = withHourCr1
# insertedCr2 = withHourCr2
# insertedCv2 = withHourCv2

# withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [insertedCr1,insertedCr2,insertedCv2], mainternanceAsCriticalSd = True)


addedData               = preData.addition_var(withIDCr1) 


handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData, headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data

# -----------------------------
# Add the cycle and RUL of system
systemData                  = pd.DataFrame()
systemData['hours']         = withTimeDataCr1["hours"]
systemData['trajectoryID']  = IdSys
withTimeDataSys             = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
dataCr1                 = withTimeDataCr1[handledValues]
cycleCr1                = withTimeDataCr1["cycle"]
rulCr1                  = withTimeDataCr1["rul"]
timeCr1                 = withTimeDataCr1["hours"]
trajectIDCr1            = withTimeDataCr1["trajectoryID"]

dataCr2                 = withTimeDataCr2[preData.crusher2Value]
cycleCr2                = withTimeDataCr2["cycle"]
rulCr2                  = withTimeDataCr2["rul"]
timeCr2                 = withTimeDataCr2["hours"]
trajectIDCr2            = withTimeDataCr2["trajectoryID"]

dataCv2                 = withTimeDataCv2[preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["cycle"]
rulCv2                  = withTimeDataCv2["rul"]
timeCv2                 = withTimeDataCv2["hours"]
trajectIDCv2            = withTimeDataCv2["trajectoryID"]

systemRUL               = withTimeDataSys["rul"]


print(timeCr1)
print(timeCr2)
print(timeCv2)
print(insertedCr1Sys["hours"])



Process:43606||2024-12-16 22:17:39,335-INFO [preProcessingData.py:loadH5File():590]:Loading H5 file ...
Process:43606||2024-12-16 22:17:40,726-INFO [preProcessingData.py:loadH5File():596]:Finish loading data from files ...
Process:43606||2024-12-16 22:17:44,585-INFO [preProcessingData.py:attachIDPerEvent():437]:Attaching ID into trajectories
Process:43606||2024-12-16 22:18:07,939-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....
Process:43606||2024-12-16 22:18:08,806-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....
Process:43606||2024-12-16 22:18:09,229-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....


0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dty

###### Crusher1

In [3]:
model       = systemModel(myConfig) 
model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]),(None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])
model.summary()
model.load_weights(model_dir = "checkpoint/sysBest_Trendability_SensorSelection/")

The shape of sysHILayer (None, 4, 5)


2024-12-16 22:18:10.176389: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-16 22:18:10.176591: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-16 22:18:10.176824: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-16 22:18:10.176902: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2024-12-16 22:18:10.176967: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)


Process:43606||2024-12-16 22:18:10,928-INFO [systemModel.py:loadWeightAndFreezePretrainedHI():984]:Loading the model at the direction: checkpoint/sysBest_Trendability_SensorSelection/


The shape of sysHILayer (None, 4, 5)


Process:43606||2024-12-16 22:18:10,962-INFO [systemModel.py:loadWeightAndFreezePretrainedHI():1023]:Loaded the model at the direction:checkpoint/sysBest_Trendability_SensorSelection/ successfully!!!


(None, 64, 41) (41,)
(None, 64, 30) (30,)
(None, 64, 31) (31,)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 41        
 ection)                                                         
                                                                 
 sensor_selection_1 (SensorS  multiple                 30        
 election)                                                       
                                                                 
 sensor_selection_2 (SensorS  multiple                 31        
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               122306    
                              (None, 4, 5)]                      
                                                         

Process:43606||2024-12-16 22:18:12,304-INFO [systemModel.py:load_weights():947]:Loading the model at the direction: checkpoint/sysBest_Trendability_SensorSelection/
Process:43606||2024-12-16 22:18:12,337-INFO [systemModel.py:load_weights():981]:Loaded the model at the direction:checkpoint/sysBest_Trendability_SensorSelection/ successfully!!!


###### System

In [4]:
results            = {}
results["cr1"]     = []
results["cr2"]     = []
results["cv2"]     = []
results["sys"]     = []
results["rul"]     = []
listRUL         = []


probsAll        = []
binaryMaskAll   = []
trajectID = IdSys.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1,seqDataCr2,seqDataCv2, seqRUL     = preData.getTensorData(dataList   =    [dataCr1, dataCr2, dataCv2, systemRUL], 
                                                                    trajectId   = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, zCr2,  zCv2,                              \
    binaryMaskCr1, binaryMaskCr2, binaryMaskCv2,    \
    probsMaskCr1, probsMaskCr2, probsMaskCv2,       \
    hiSystem, predictedRul                          = model.getHIandRUL([seqDataCr1,seqDataCr2,seqDataCv2])
    results["cr1"].append(zCr1)
    results["cr2"].append(zCr2)
    results["cv2"].append(zCv2)
    results["sys"].append(hiSystem)
    results["rul"].append(predictedRul)
    listRUL.append(seqRUL[:,-1,:])


Process:43606||2024-12-16 22:18:12,351-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


0
(6123, 64, 41) (41,)
(6123, 64, 30) (30,)
(6123, 64, 31) (31,)


Process:43606||2024-12-16 22:18:13,858-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


1


Process:43606||2024-12-16 22:18:14,293-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


(1113, 64, 41) (41,)
(1113, 64, 30) (30,)
(1113, 64, 31) (31,)
2
(16540, 64, 41) (41,)
(16540, 64, 30) (30,)
(16540, 64, 31) (31,)


... (194 outputs left out)

Process:43606||2024-12-16 22:20:30,902-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


112
(10047, 64, 41) (41,)
(10047, 64, 30) (30,)
(10047, 64, 31) (31,)


Process:43606||2024-12-16 22:20:33,749-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


113
(5996, 64, 41) (41,)
(5996, 64, 30) (30,)
(5996, 64, 31) (31,)


In [5]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

# Example usage

print(listRUL[0].shape, results["rul"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rul"])
print("Overall MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))

temp_df = pd.concat([rulCr1, rulCr2, rulCv2], axis=1)
# print(np.max(np.concatenate(rulCr1, rulCr2 , rulCv2)), np.min(np.concatenate(rulCr1 , rulCr2, rulCv2)))
print(temp_df.max().max(),temp_df.min().min())





(6123, 1) (6123, 1)
Overall MAE: 52.323296
454.7669999999998 0.0
670.55 0.0
